# 01 — Acquisition and Integration

**Research question (T9):** Do richer and more populous countries win more medals *per person*, and do host countries have an advantage?

## Data sources
| Source | File | Key columns |
|--------|------|-------------|
| Kaggle — 120 Years of Olympic History | `athlete_events.csv` | ID, Name, NOC, Year, Season, Games, Event, Medal |
| Kaggle — NOC → region lookup | `noc_regions.csv` | NOC, region, notes |
| World Bank WDI API | `wdi_indicators.csv` | iso3, country, year, indicator_code, value |

WDI indicators fetched: `NY.GDP.PCAP.CD` (GDP per capita, current USD) and `SP.POP.TOTL` (total population).  
To reproduce: run `python src/fetch_data.py` (calls the World Bank REST API via curl).

## Integration pipeline (`src/build_analysis.py`)

```
athlete_events.csv
  └─ filter: Year ≥ 1960, Medal is not NaN
  └─ deduplicate: drop_duplicates([Games, NOC, Event, Medal])  ← team-medal fix
  └─ groupby [Year, Season, Games, NOC] → medal_counts
       |
       ├─ merge ← noc_regions (NOC → region → ISO3 crosswalk)
       |           └─ ISO3 derived via: WDI country-name match → documented alias → NOC direct → historical sensitivity map
       |
       └─ merge ← WDI macro (ISO3 + Year → gdp_per_capita_usd, population)
           |
           └─ feature engineering:
               medals_per_million  = medals / population × 1 000 000
               log_gdp_per_capita  = log1p(gdp_per_capita_usd)
               log_population      = log1p(population)
               log_medals          = log1p(medals)
               gold_share          = gold / medals
               host_flag           = 1 if NOC hosted that edition
               income_quartile     = qcut(gdp_per_capita_usd, 4)
```

### Hard parts addressed
1. **Team-medal deduplication** — `drop_duplicates(['Games','NOC','Event','Medal'])` collapses relay/team events so each medal counts once per NOC, not once per athlete on the squad.
2. **Historical NOC mapping** — URS→RUS, GDR→DEU, FRG→DEU, EUN→RUS, YUG→SRB, TCH→CZE, SCG→SRB.  These are labelled *sensitivity mappings* in the crosswalk and are **not** silently applied to raw files.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'tmp' / 'matplotlib'))

from build_analysis import main
main()

### Load and inspect the integrated analytical dataset

In [ ]:
import pandas as pd

integrated = pd.read_csv(ROOT / 'data/processed/olympics_wealth_country_edition.csv')
join_audit = pd.read_csv(ROOT / 'data/processed/join_audit.csv')

print('Integrated country-edition rows:', len(integrated))
print('Year range:', integrated.Year.min(), '—', integrated.Year.max())
print('Unique NOCs in analytical sample:', integrated.NOC.nunique())
print()
print('=== Join audit ===')
display(join_audit)
print()
print('=== First 5 rows ===')
display(integrated.head())

### NOC → ISO3 crosswalk: mapping sources

In [ ]:
crosswalk = pd.read_csv(ROOT / 'data/processed/noc_to_iso3_crosswalk.csv')
print('Mapping source breakdown:')
display(crosswalk.mapping_source.value_counts().reset_index())

print('\nHistorical sensitivity mappings (review carefully):')
display(crosswalk[crosswalk.mapping_source == 'historical sensitivity mapping'][['NOC','region','iso3','mapping_source']])

### Deduplication impact: team-medal rows reduced

In [ ]:
import json
results = json.loads((ROOT / 'data/processed/analysis_results.json').read_text())
s = results['sample']
print(f"Athlete-level medal rows (before dedup): {s['athlete_medal_rows_before_dedup']:,}")
print(f"Distinct medals after dedup:              {s['team_and_individual_medal_rows_after_dedup']:,}")
reduction = (1 - s['team_and_individual_medal_rows_after_dedup'] / s['athlete_medal_rows_before_dedup']) * 100
print(f"Reduction: {reduction:.1f}% — these were team athletes counted once per sport, now counted once per medal")